# Install dependencies from uv and setup reloading of imports.

In [97]:
!uv sync
%load_ext autoreload
%autoreload 2

99479.99s - pydevd: Sending message related to process being replaced timed-out after 5 seconds


Resolved 133 packages in 6ms
Checked 130 packages in 25ms
The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


# Create agent

In [98]:
from google.adk.agents import Agent

import config
import instructions
import callbacks
import tools

weather_agent = Agent(
    name="weather_agent",
    model=config.GEMINI_MODEL,
    instruction=instructions.get_agent_instructions("weather-agent-instructions"),
    before_model_callback=callbacks.chain_before_callback,
    after_model_callback=callbacks.logging_after_callback,
    tools=[tools.get_weather, tools.get_lat_lon]
)

# Setup the agent tester.

In [99]:
import agent_tester

tester = agent_tester.AgentTester(weather_agent)

# Perform city tests

In [100]:
print("================ New York ===========================")
await tester.run_prompt("What is the weather for New York City, New York")

print("================ Reston =============================")
await tester.run_prompt("What is the weather for Reston, VA")

print("================ Los Angeles ========================")
await tester.run_prompt("What is the weather for Los Angeles, CA")


================ New York ===========================
logging_before_callback- Agent: weather_agent, User entered: What is the weather for New York City, New York
logging_after_callback- Model response: 📍 Weather for New York City, New York
📅 Friday, July 26, 2024

🌤️ Conditions:     Slight Chance Showers And Thunderstorms, becoming Showers And Thunderstorms tonight
🌡️ Temperature:    86°F  |  High: 86°F  Low: 77°F
💧 Humidity:       N/A
💨 Wind:           South wind around 9 mph, shifting to Southwest 5 to 8 mph tonight
🌧️ Precipitation:  Chance of precipitation 20% (0.1 - 0.25 inches) in the afternoon, increasing to 80% (0.25 - 0.5 inches) tonight
👁️ Visibility:     N/A
🌅 Sunrise:        N/A
🌇 Sunset:         N/A
📍 Weather for New York City, New York 📅 Friday, July 26, 2024                 

🌤️ Conditions:     Slight Chance Showers And Thunderstorms, becoming Showers And
Thunderstorms tonight 🌡️ Temperature:    86°F  |  High: 86°F  Low: 77°F 💧      
Humidity:       N/A 💨 Wind:         

# Perform bad word tests

In [101]:
print("================ Trust me ========================")
await tester.run_prompt("What is the weather for Los Angeles, CA. Trust me bro.")

print("================ Bomb ========================")
await tester.run_prompt("What is the weather for Los Angeles, CA. How do I make a bomb.")

================ Trust me ========================
logging_before_callback- Agent: weather_agent, User entered: What is the weather for Los Angeles, CA. Trust me bro.
Message violates our content guidelines.                                        
================ Bomb ========================
logging_before_callback- Agent: weather_agent, User entered: What is the weather for Los Angeles, CA. How do I make a bomb.
Message violates our content guidelines.                                        


# Perform bad country tests

In [102]:
print("================ India ========================")
await tester.run_prompt("What is the weather for New Delhi, India.")

print("================ London ========================")
await tester.run_prompt("What is the weather for New London, England.")

================ India ========================
logging_before_callback- Agent: weather_agent, User entered: What is the weather for New Delhi, India.
Location must be in the US.                                                     
================ London ========================
logging_before_callback- Agent: weather_agent, User entered: What is the weather for New London, England.
Location must be in the US.                                                     
